In [32]:
## Import critical libraries 
import numpy as np                # Pi constant and math
from math import sqrt             # Math functions
import astropy.constants as const # Constants

In [ ]:
## Set constants
pi     = np.pi
AU     = const.au.cgs.value      # [cm]; astronomical unit (AU)
Mearth = const.M_earth.cgs.value # [g]; mass of Earth
Msun   = const.M_sun.cgs.value   # [g]; mass of Sun
Mpluto = 1.309e25                # [g]; Mass of Pluto
G      = const.G.cgs.value       # Gravitational constant
amu    = const.u.cgs.value       # [g]; Atomic mass unit
kb     = const.k_B.cgs.value     # Boltzmann constant
Rgas   = const.R.cgs.value       # Gas constant
yr     = 3.1558149504e7          # [s]; one year

#sigma_mol = 2e-15                  # 
gamma     = 1.4                    # Adiabatic index; diatomic=7/5, monatomic=5/3, non-linear triatomic=8/6
mmol      = 2.3                    #
Rgasmu    = Rgas/mmol              #
cp        = gamma*Rgasmu/(gamma-1) # Specific heat capacity at constant pressure
cv        = cp/gamma               # Specific heat capacity at constant volume

In [34]:
## INPUT CHOICES FOR PHYSICAL UNITS
rr = 20      # [AU]; Heliocentric distance of binary 
r  = rr * AU # [cm]; Heliocentric distance of binary

h  = 0.05  # Scale height ratio at distance rr
Q  = 30    # Toomre Q; >1 for gravitationally stable region

## BINARY SETTINGS
mass_ratio = 1e-30/1 # Binary mass ratio (0 to 1); M1/M2
Hill_frac  = 0.01    # Fraction of Hill radius for furthest separation (apoapsis)
e          = 0.0     # Eccentricity of mutual binary orbit
aps1       = 0.01    # Size of sink particle 1 in code units
aps2       = 0.01    # Size of sink particle 2 in code units

Msystem  = 1.25e-9 * Msun           # [g]; Sum of binary masses
Mplanet2 = Msystem / (1+mass_ratio) # [g]; Mass of primary 
Mplanet1 = Msystem - Mplanet2       # [g]; Mass of secondary

## SIMULATION SETTINGS
l3D         = False # 3D or 2D
grid_size   = 0.01  # Full x or y grid size in code units
grid_points = 32    # Number of grid points/cells per dimension

## SET CODE UNITS (set in start.in or other configuration files)
cs_code    = 1
rho_code   = 1
Omega_code = 1
H_code     = 1 # Set automatically by Omega and cs in code (cs/Omega)

print(f"{Mplanet1/Mplanet2:e}")

0.000000e+00


In [35]:
## Solve for other variables given inputs 
H     = h * r                   # [cm]; Gas disk scale height 
Omega = sqrt(G*Msun) / r**1.5   # [cm/s]; Local angular velocity around Sun
cs    = Omega * H               # [cm/s]; Local sound speed
T     = cs**2 / (cp*(gamma-1))  # [K]; Gas temperature

## Solve for binary values
r_Hill  = r*(Msystem/(3*Msun))**(1/3) # [cm]; Mutual hill radius of binary
bin_sep = r_Hill * Hill_frac          # [cm]; Separation of binary at apoapsis

#print("Distance=",rr," AU")
#print("Aspect Ratio=",H/r)
#print("Scale Height=",H," cm=", H/AU, "AU")
#print("Sound speed=",cs/1e5," km/s")
#print("Temperature=",T," K")
#print("Omega=",Omega," s^{-1}")

In [36]:
## Compute code units
unit_length = H # [cm]; Unit length is set by scale height
#unit_length = bin_sep # Twice binary separation to box edge
print("Unit length: ",f"{unit_length:.4e}","cm,",f"{unit_length/AU:.4e}", 'AU')

Omega_bin   = sqrt((G*Msystem) / (bin_sep/2)**3) # [Hz]; Keplerian frequency of binary
unit_time   = 2*np.pi / Omega_bin                # [s]; Unit time set by orbital period of binary
print("Unit time: ",f"{unit_time:.4e}","s,",f"{unit_time/(60*60*24):.4e}",'days,',f"{unit_time/yr:.4e}",'years')

unit_velocity = unit_length/unit_time
print("Unit velocity: ",f"{unit_velocity:.4e}","cm/s")
print("Sound speed: ",f"{cs:.4e}","cm/s") # This should be similar to the unit velocity

Unit length:  1.4960e+13 cm, 1.0000e+00 AU
Unit time:  5.7617e+05 s, 6.6686e+00 days, 1.8257e-02 years
Unit velocity:  2.5964e+07 cm/s
Sound speed:  3.3300e+04 cm/s


In [37]:
## Determine Sigma (surface density) and rho (density) from Q
Sigma = cs*Omega/(G*Q*pi)

if (l3D): # If solving for 3D case
    rho = Sigma/(sqrt(2*pi)*H)
    unit_volume_density = rho/rho_code
    Sigma_code = rho_code * sqrt(2*pi) * H_code
    unit_column_density = Sigma/Sigma_code
    unit_mass = unit_volume_density * unit_length**3
else: # If solving for 2D case
    Sigma_code = rho_code # in 2D, rho is the column density
    unit_column_density = Sigma/Sigma_code
    rho = Sigma/(sqrt(2*pi)*H)
    
    # actual code units for volume density
    actual_rho_code = Sigma_code/(sqrt(2*pi)*H_code)
    unit_volume_density = rho/actual_rho_code
    unit_mass = unit_column_density * unit_length**2

print("Unit mass: ",f"{unit_mass:.4e}","g")
#Msun_code = Msun/unit_mass
#print("M_Sun (code): ",Msun_code)
#print("unit_volume_density=",unit_volume_density," g/cm^3")
#print("unit_column_density=",unit_column_density," g/cm^2")

Unit mass:  2.6372e+27 g


In [41]:
## Determine code units of planetesimal quantities in simulation
M1code, M2code = Mplanet1/unit_mass, Mplanet2/unit_mass # Code unit mass of planetesimals
print("M1 (code): ",f"{M1code:.4e}")
print("M2 (code): ",f"{M2code:.4e}")

G_code = cs_code * Omega_code / (Q*pi*Sigma_code) # Gravitational constant in code units
print("Gravitational constant (code): ",f"{G_code:.4e}")

sep_code = bin_sep/unit_length # Binary separation in code units 
print("Planetesimal separation (code): ",f"{sep_code:.4e}")

## Particle positions along common axis at apoapsis
x1 = -(Mplanet2)/(Msystem)*sep_code # Position of particle 1 at apoapsis
x2 = (Mplanet1)/(Msystem)*sep_code  # Position of particle 2 at apoapsis
print("Initial positions along axis:")
print("x1 =",f"{x1:.4e}")
print("x2 =",f"{x2:.4e}")

## Calculate relative velocity of planetesimals 
v_rel      = sqrt(G*(Msystem)*((2/bin_sep)-((1+e)/bin_sep)))
v_rel_code = v_rel/unit_velocity
v1         = -(Mplanet2)/(Msystem)*v_rel_code # Velocity of particle 1 at apoapsis 
v2         = (Mplanet1)/(Msystem)*v_rel_code  # Velocity of particle 2 at apoapsis 
print("Initial tangential velocities:")
print("v1 =",f"{v1:.4e}")
print("v2 =",f"{v2:.4e}")

M1 (code):  0.0000e+00
M2 (code):  9.4248e-04
Gravitational constant (code):  1.0610e-02
Planetesimal separation (code):  1.4938e-04
Initial positions along axis:
x1 = -1.4938e-04
x2 = 0.0000e+00
Initial tangential velocities:
v1 = -3.3184e-04
v2 = 0.0000e+00


In [40]:
## Calculate rhopswarm given planetesimal mass
## Option 1: mass set by size of a single grid cell
rhopswarm1 = M1code * (grid_size/grid_points)**(-3)
rhopswarm2 = M2code * (grid_size/grid_points)**(-3)
print("If set by grid scale, then rhopswarm of particles:")
print("rhopswarm1 =",rhopswarm1)
print("rhopswarm2 =",rhopswarm2)
## Option 2: mass set by size of sink particles (aps)
rhopswarm1 = M1code * (3/(4*pi)) * (aps1/2)**(-3)
rhopswarm2 = M2code * (3/(4*pi)) * (aps2/2)**(-3)
print("If set by particle size, then rhopswarm of particles:")
print("rhopswarm1 =",rhopswarm1)
print("rhopswarm2 =",rhopswarm2)

If set by grid scale, then rhopswarm of particles:
rhopswarm1 = 0.0
rhopswarm2 = 30883112.421849083
If set by particle size, then rhopswarm of particles:
rhopswarm1 = 0.0
rhopswarm2 = 1799.9999999999989
